# SEC MD&A Pipeline — Table Extraction → Chunking → Embeddings → Redis Vector Search

**Reconstructed from `test_secs_chunks.html` (Jupyter export)**  
Pipeline steps:
1. Load SEC MD&A data from Parquet  
2. Extract tables (HTML + whitespace-aligned) → SQLite  
3. Chunk cleaned text with MD&A-section awareness (15 % overlap)  
4. Generate sentence-transformer embeddings  
5. Store embeddings in **Redis Vector Search** (RediSearch / Redis Stack)  
6. Run a sample KNN query  

### Prerequisites
```
pip install sentence-transformers redis pandas pyarrow beautifulsoup4
```
Redis Stack must be running, e.g.:
```bash
docker run -d -p 6379:6379 redis/redis-stack-server:latest
```

In [ ]:
# ── 0. Install dependencies (run once) ────────────────────────────────────
import subprocess, sys

PACKAGES = [
    "sentence-transformers",
    "redis",
    "pandas",
    "pyarrow",
    "beautifulsoup4",
    "numpy",
]

for pkg in PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages ready.")

In [ ]:
# ── 1. Imports & path configuration ───────────────────────────────────────
import os, sys, json, re, sqlite3, struct, uuid
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd

# ── Adjust these paths to match your environment ──────────────────────────
# Directory containing table_extraction.py, text_chunking.py, embeddings.py
PIPELINE_DIR = Path("/home/d1ttb/mda_wip/mda_pipeline")

# Parquet file with SEC MD&A data
PARQUET_PATH = Path("/home/d1ttb/mda_wip/mda_merged_sic_naics_24_25.parquet")

# HTML notebook export (used to demonstrate HTML-based table extraction)
HTML_PATH    = Path("test_secs_chunks.html")   # adjust if needed

# SQLite output for extracted tables
SQLITE_PATH  = "sec_tables.db"

# Redis connection
REDIS_HOST   = "localhost"
REDIS_PORT   = 6379
REDIS_DB     = 0
REDIS_INDEX  = "sec_mda_chunks"       # RediSearch index name
REDIS_PREFIX = "sec:chunk:"           # key prefix for stored hashes

# Embedding / chunking settings
EMBED_MODEL       = "multi-qa-MiniLM-L6-cos-v1"  # dim = 384
CHUNK_SIZE_TOKENS = 128
OVERLAP_PERCENT   = 15
BATCH_SIZE        = 64
# ─────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(PIPELINE_DIR))

print(f"Pipeline dir : {PIPELINE_DIR}")
print(f"Parquet path : {PARQUET_PATH}")
print(f"Redis target : {REDIS_HOST}:{REDIS_PORT}  index={REDIS_INDEX}")

## Part A — Extract code cells & table outputs from the HTML notebook

The cell below parses `test_secs_chunks.html` (an exported Jupyter notebook) and pulls out:
- All **code cells** (the source `<pre>` blocks stripped of HTML spans)
- All **table outputs** already rendered in the notebook (structured as `ExtractedTable`-style dicts)

In [ ]:
# ── 2. Parse the HTML notebook export ─────────────────────────────────────
from bs4 import BeautifulSoup

def strip_spans(tag) -> str:
    """Return the plain text of a syntax-highlighted <pre> element."""
    return tag.get_text()


def parse_html_notebook(html_path: Path) -> Dict:
    """
    Walk an nbconvert-generated HTML file and collect:
      - code_cells : list of source code strings (one per In[N] cell)
      - table_outputs : list of table-like output blocks (text pre elements
                        whose content starts with table row markers)
    """
    soup = BeautifulSoup(html_path.read_text(encoding="utf-8"), "html.parser")

    code_cells   = []
    table_outputs = []
    text_outputs  = []

    for cell_div in soup.select("div.jp-CodeCell"):
        # ── source code ──────────────────────────────────────────────────
        input_pre = cell_div.select_one("div.jp-InputArea pre")
        prompt    = cell_div.select_one("div.jp-InputPrompt")
        label     = prompt.get_text(strip=True) if prompt else ""
        src       = strip_spans(input_pre).strip() if input_pre else ""
        if src:
            code_cells.append({"label": label, "source": src})

        # ── outputs ───────────────────────────────────────────────────────
        for out_pre in cell_div.select("div.jp-OutputArea pre"):
            raw = out_pre.get_text()
            # Heuristic: lines starting with '──' or 'Table' are table blocks
            if re.search(r"(?m)^\s*──\s*Table|Table\s+\d+", raw):
                table_outputs.append({"cell": label, "raw": raw.strip()})
            else:
                text_outputs.append({"cell": label, "raw": raw.strip()})

    return {
        "code_cells":    code_cells,
        "table_outputs": table_outputs,
        "text_outputs":  text_outputs,
    }


if HTML_PATH.exists():
    nb = parse_html_notebook(HTML_PATH)
    print(f"Code cells found   : {len(nb['code_cells'])}")
    print(f"Table outputs found: {len(nb['table_outputs'])}")
    print(f"Text outputs found : {len(nb['text_outputs'])}")
    print("\n── Extracted code cell labels:")
    for cc in nb["code_cells"]:
        print(f"  {cc['label']:10s}  {cc['source'][:60].strip()!r}")
else:
    print(f"[warn] HTML file not found at {HTML_PATH} — skipping HTML parse.")
    nb = {"code_cells": [], "table_outputs": [], "text_outputs": []}

In [ ]:
# ── 3. Show tables captured from HTML notebook outputs ────────────────────
for i, tbl in enumerate(nb["table_outputs"]):
    print(f"\n{'='*60}")
    print(f"  HTML-output table block {i}  (from cell {tbl['cell']})")
    print('='*60)
    print(tbl["raw"][:800])
    if len(tbl["raw"]) > 800:
        print(f"  ... ({len(tbl['raw'])-800} more chars)")

## Part B — Full pipeline: Parquet → Tables + Chunks → Embeddings → Redis

In [ ]:
# ── 4. Load MD&A data from Parquet ────────────────────────────────────────
from table_extraction import TableExtractor, ExtractedTable, SQLiteTableStore
from text_chunking    import TextChunker, TextChunk
from embeddings       import EmbeddingGenerator, EmbeddedChunk


def divider(title: str) -> None:
    print(f"\n{'='*70}\n  {title}\n{'='*70}")


# Simple dataclass-compatible wrapper so the pipeline classes work with
# a plain DataFrame row (no full SECDocument dataclass required).
class MDAdoc:
    __slots__ = [
        "document_id", "accession", "year", "cik", "name",
        "sic", "ein", "address", "naics3", "naics17Title",
        "is_ambiguous_naics3", "mda_text", "metadata",
    ]
    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)


if PARQUET_PATH.exists():
    df_mda = pd.read_parquet(PARQUET_PATH)
    print(f"Loaded {len(df_mda):,} rows from Parquet")
    print(f"Columns: {list(df_mda.columns)}")

    # Build document objects — process a configurable slice for demo/testing
    MAX_DOCS = 50        # set to None to process all
    subset   = df_mda.iloc[:MAX_DOCS] if MAX_DOCS else df_mda

    documents = []
    for _, row in subset.iterrows():
        doc = MDAdoc(
            document_id          = f"{row['cik']}_{row['year']}_{row['accession']}",
            accession            = row.get("accession", ""),
            year                 = str(row.get("year", "")),
            cik                  = str(row.get("cik", "")),
            name                 = row.get("name", ""),
            sic                  = str(row.get("sic", "")),
            ein                  = str(row.get("ein", "")),
            address              = row.get("address", ""),
            naics3               = str(row.get("naics3", "")),
            naics17Title         = row.get("naics17Title", ""),
            is_ambiguous_naics3  = bool(row.get("is_ambiguous_naics3", False)),
            mda_text             = row.get("mda_text", ""),
            metadata             = {},
        )
        documents.append(doc)

    print(f"Prepared {len(documents):,} MDAdoc objects")

else:
    print(f"[warn] Parquet not found at {PARQUET_PATH}")
    print("       Creating a synthetic single-document demo instead...")

    DEMO_TEXT = """
ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS

Overview

The company delivered strong results in fiscal 2024 with revenue growth of 12 %
and expanded operating margins. Our diversified portfolio of utility assets
continued to generate stable cash flows.

Results of Operations

Revenue increased to $4.0 billion, up from $3.6 billion in the prior year.
Operating income rose to $1.1 billion, driven by lower fuel costs and higher
transmission volumes.

<table>
<tr><th>Year</th><th>Revenue ($M)</th><th>Op. Income ($M)</th></tr>
<tr><td>2024</td><td>4,000</td><td>1,100</td></tr>
<tr><td>2023</td><td>3,600</td><td>  980</td></tr>
<tr><td>2022</td><td>3,200</td><td>  870</td></tr>
</table>

Liquidity and Capital Resources

Cash provided by operations totalled $1.3 billion. Capital expenditures were
$1.5 billion, funding infrastructure upgrades and renewable-energy additions.
We maintain $2.0 billion of revolving credit capacity.

Cash Flows

Operating cash flow improved $325 million year-over-year. Investing outflows
reflect our five-year $9.8 billion capital plan.

Risk Factors

Material risks include regulatory changes, weather variability, interest-rate
exposure, and evolving environmental standards. We mitigate these through
diversified fuel sourcing, hedging programs, and proactive regulatory engagement.

Environmental Matters

We are committed to reducing carbon emissions 80 % by 2030 relative to 2005
baseline levels. Capital investments in solar, wind, and battery storage
support this trajectory.
"""

    demo_doc = MDAdoc(
        document_id="DEMO_0000001_2024", accession="0000001-24-000001",
        year="2024", cik="0000001", name="Demo Utility Corp",
        sic="4911", ein="12-3456789", address="123 Main St",
        naics3="221", naics17Title="Utilities",
        is_ambiguous_naics3=False, mda_text=DEMO_TEXT, metadata={},
    )
    documents = [demo_doc]
    print(f"Demo document created ({len(DEMO_TEXT):,} chars)")

In [ ]:
# ── 5. Table extraction ───────────────────────────────────────────────────
divider("STEP 2 — TABLE EXTRACTION")

extractor = TableExtractor(min_formatted_rows=3)
cleaned_docs, all_tables = extractor.extract_from_documents(documents)

print(f"\n  Documents processed : {len(documents):,}")
print(f"  Tables extracted    : {len(all_tables):,}")

# Preview first 3 tables
for i, t in enumerate(all_tables[:3]):
    print(f"\n  ── Table {i} ──")
    print(f"     ID    : {t.table_id}")
    print(f"     Title : {t.table_title}")
    print(f"     Rows  : {len(t.table_data)}")
    for row in t.table_data[:4]:
        print(f"       {row}")
    if len(t.table_data) > 4:
        print(f"       ... ({len(t.table_data)-4} more rows)")

In [ ]:
# ── 6. Persist tables to SQLite ────────────────────────────────────────────
divider("STEP 2b — PERSIST TABLES TO SQLITE")

with SQLiteTableStore(db_path=SQLITE_PATH) as store:
    n_stored = store.store_tables(all_tables)
    stats    = store.statistics()

print("\nSQLite statistics:")
for k, v in stats.items():
    print(f"  {k}: {v}")

In [ ]:
# ── 7. Text chunking ──────────────────────────────────────────────────────
divider("STEP 3 — TEXT CHUNKING (MD&A structure-aware)")

chunker = TextChunker(
    chunk_size_tokens=CHUNK_SIZE_TOKENS,
    overlap_percent=OVERLAP_PERCENT,
)
chunks = chunker.run(cleaned_docs, strategy="mda")

# Section coverage summary
divider("SECTION COVERAGE SUMMARY")
seen: Dict[str, Dict] = {}
for c in chunks:
    name = c.section_name or "(no section)"
    if name not in seen:
        seen[name] = {"chunks": 0, "chars": 0}
    seen[name]["chunks"] += 1
    seen[name]["chars"]  += len(c.text)

print(f"  {'Section':<45s} {'Chunks':>6s} {'Chars':>9s}")
print(f"  {'-'*45} {'-'*6} {'-'*9}")
for name, info in sorted(seen.items(), key=lambda x: -x[1]['chunks']):
    print(f"  {name:<45s} {info['chunks']:>6d} {info['chars']:>9,d}")

total_chars = sum(len(c.text) for c in chunks)
print(f"\n  Total chunks: {len(chunks):,}   Total chars: {total_chars:,}")

In [ ]:
# ── 8. Overlap verification ────────────────────────────────────────────────
divider("OVERLAP VERIFICATION")
print(f"  chunk_size   = {chunker.chunk_size} chars (~{chunker.chunk_size_tokens} tokens)")
print(f"  overlap      = {chunker.overlap} chars ({chunker.overlap_percent}%)")

overlap_ok   = True
gap_count    = 0
sample_shown = 0

for i in range(1, len(chunks)):
    prev, cur = chunks[i-1], chunks[i]
    if prev.section_name != cur.section_name:
        continue                                  # different sections — gap is expected
    if prev.source_document_id != cur.source_document_id:
        continue
    overlap_chars = prev.end_char - cur.start_char
    if overlap_chars < 0:
        gap_count += 1
        overlap_ok = False
        if sample_shown < 5:
            print(f"  ⚠ GAP chunks {i-1}→{i} ({prev.section_name}): {-overlap_chars} chars")
            sample_shown += 1

if overlap_ok:
    print("\n  ✅  All same-section adjacent chunks overlap correctly.")
else:
    print(f"\n  ❌  {gap_count} chunk pairs have gaps — review boundary logic.")

In [ ]:
# ── 9. Generate embeddings ─────────────────────────────────────────────────
divider("STEP 4 — EMBEDDING GENERATION")

embedder = EmbeddingGenerator(
    model_name=EMBED_MODEL,
    normalize_embeddings=True,
)

embedded_chunks: List[EmbeddedChunk] = embedder.run(chunks, batch_size=BATCH_SIZE)

print(f"\n  Embedded chunks     : {len(embedded_chunks):,}")
print(f"  Embedding dimension : {embedder.embedding_dim}")

# Quick sanity check — all dims consistent, values finite
ok = embedder.verify()
print(f"  Verify passed       : {ok}")

# Statistics
print("\n  Statistics:")
for k, v in embedder.statistics().items():
    print(f"    {k}: {v}")

## Part C — Redis Vector Search

We use **Redis Stack** (RediSearch module) to:
1. Create a flat/HNSW vector index over the embedding field  
2. Store each `EmbeddedChunk` as a Redis Hash  
3. Query with `FT.SEARCH … VECTOR RANGE / KNN` syntax  

Connection details are configured in cell 1 via `REDIS_HOST`, `REDIS_PORT`, `REDIS_INDEX`, `REDIS_PREFIX`.

In [ ]:
# ── 10. Connect to Redis and (re)create the vector index ──────────────────
import redis
from redis.commands.search.field  import TextField, TagField, NumericField, VectorField
from redis.commands.search.indexDefinition import IndexDefinition, IndexType
from redis.commands.search.query  import Query

r = redis.Redis(host=REDIS_HOST, port=REDIS_PORT, db=REDIS_DB, decode_responses=False)

try:
    info = r.ping()
    print(f"Redis PING → {info}")
except redis.exceptions.ConnectionError as exc:
    print(f"[ERROR] Cannot reach Redis at {REDIS_HOST}:{REDIS_PORT}: {exc}")
    print("Make sure Redis Stack is running (see Prerequisites above).")
    raise

VECTOR_DIM = embedder.embedding_dim   # e.g. 384 for MiniLM

# Drop existing index if present (idempotent re-runs)
try:
    r.ft(REDIS_INDEX).dropindex(delete_documents=False)
    print(f"Dropped existing index '{REDIS_INDEX}'")
except Exception:
    pass  # index didn't exist yet

# ── Schema ────────────────────────────────────────────────────────────────
schema = [
    TextField("chunk_id"),
    TextField("source_document_id"),
    TextField("section_name"),
    TextField("text"),
    TagField("embedding_model"),
    TagField("year"),
    TagField("cik"),
    TagField("naics3"),
    TagField("sic"),
    NumericField("chunk_index"),
    NumericField("total_chunks"),
    NumericField("start_char"),
    NumericField("end_char"),
    VectorField(
        "embedding",
        "HNSW",                   # use FLAT for small datasets (<100k)
        {
            "TYPE":            "FLOAT32",
            "DIM":             VECTOR_DIM,
            "DISTANCE_METRIC": "COSINE",
            "M":               16,
            "EF_CONSTRUCTION": 200,
        },
    ),
]

index_def = IndexDefinition(
    prefix=[REDIS_PREFIX],
    index_type=IndexType.HASH,
)

r.ft(REDIS_INDEX).create_index(schema, definition=index_def)
print(f"Created RediSearch index '{REDIS_INDEX}' (dim={VECTOR_DIM}, HNSW, COSINE)")

In [ ]:
# ── 11. Store all embedded chunks in Redis ────────────────────────────────
divider("STEP 5 — STORING EMBEDDINGS IN REDIS")


def vec_to_bytes(vec: List[float]) -> bytes:
    """Pack a float list into a little-endian FLOAT32 byte array for Redis."""
    return struct.pack(f"{len(vec)}f", *vec)


pipe = r.pipeline(transaction=False)
PIPE_BATCH = 500

stored = 0
for ec in embedded_chunks:
    key = f"{REDIS_PREFIX}{ec.chunk_id}"
    meta = ec.metadata or {}

    mapping = {
        "chunk_id":          ec.chunk_id,
        "source_document_id": ec.source_document_id,
        "section_name":      ec.section_name or "",
        "text":              ec.text,
        "embedding_model":   ec.embedding_model,
        # metadata fields (safe string conversion)
        "year":              str(meta.get("year",  "")),
        "cik":               str(meta.get("cik",   "")),
        "naics3":            str(meta.get("naics3","")),
        "sic":               str(meta.get("sic",   "")),
        "chunk_index":       int(meta.get("chunk_index",  ec.metadata.get("chunk_index", 0))),
        "total_chunks":      int(meta.get("total_chunks", ec.metadata.get("total_chunks", 1))),
        "start_char":        int(meta.get("start_char",   ec.metadata.get("start_char", 0))),
        "end_char":          int(meta.get("end_char",     ec.metadata.get("end_char", 0))),
        # binary vector — must NOT be decoded as text
        "embedding":         vec_to_bytes(ec.embedding),
    }

    pipe.hset(key, mapping=mapping)
    stored += 1

    if stored % PIPE_BATCH == 0:
        pipe.execute()
        pipe = r.pipeline(transaction=False)
        print(f"  Stored {stored:,}/{len(embedded_chunks):,} …")

pipe.execute()  # flush remainder
print(f"\n  ✅ Stored {stored:,} chunks in Redis under prefix '{REDIS_PREFIX}'")

In [ ]:
# ── 12. Confirm index info ────────────────────────────────────────────────
info = r.ft(REDIS_INDEX).info()

print(f"Index name         : {info.get('index_name', b'').decode() if isinstance(info.get('index_name'), bytes) else info.get('index_name', '')}")
print(f"Num docs indexed   : {info.get('num_docs', 0)}")
print(f"Num records        : {info.get('num_records', 0)}")
print(f"Indexing failures  : {info.get('hash_indexing_failures', 0)}")

# Key count sanity-check
n_keys = len(r.keys(f"{REDIS_PREFIX}*"))
print(f"Redis keys matched : {n_keys:,}")

In [ ]:
# ── 13. Sample KNN vector search query ────────────────────────────────────
divider("STEP 6 — KNN VECTOR SEARCH DEMO")

QUERY_TEXT = "liquidity and capital resources cash flow"
TOP_K      = 5

# Encode the query with the same model
query_vec = embedder.encode_one(QUERY_TEXT)
query_bytes = vec_to_bytes(query_vec)

q = (
    Query(f"*=>[KNN {TOP_K} @embedding $vec AS score]")
    .sort_by("score")
    .return_fields("score", "chunk_id", "section_name", "text", "year", "cik")
    .dialect(2)
)

results = r.ft(REDIS_INDEX).search(q, query_params={"vec": query_bytes})

print(f"Query       : {QUERY_TEXT!r}")
print(f"Top-{TOP_K} results:")
print()

for rank, doc in enumerate(results.docs, start=1):
    score   = getattr(doc, "score",        "N/A")
    section = getattr(doc, "section_name", "")
    cik     = getattr(doc, "cik",          "")
    year    = getattr(doc, "year",         "")
    text    = getattr(doc, "text",         "")
    snippet = (text[:150].replace("\n", " ") + "…") if len(text) > 150 else text

    print(f"  #{rank}  score={score}  [{section}]  cik={cik}  year={year}")
    print(f"       {snippet}")
    print()

In [ ]:
# ── 14. Filtered KNN search (by section + year tag) ───────────────────────
divider("FILTERED KNN SEARCH (section + year)")

FILTER_SECTION = "Results Of Operations"
FILTER_YEAR    = "2024"
FILTER_TOPK    = 3

# Full-text pre-filter on section_name + year tag; then KNN re-rank
filter_str = f"(@section_name:{{{FILTER_SECTION}}}) @year:{{{FILTER_YEAR}}}"

fq = (
    Query(f"({filter_str})=>[KNN {FILTER_TOPK} @embedding $vec AS score]")
    .sort_by("score")
    .return_fields("score", "chunk_id", "section_name", "text", "year", "cik")
    .dialect(2)
)

try:
    fresults = r.ft(REDIS_INDEX).search(fq, query_params={"vec": query_bytes})
    print(f"Filter: section='{FILTER_SECTION}'  year='{FILTER_YEAR}'")
    print(f"Top-{FILTER_TOPK} results:")
    for rank, doc in enumerate(fresults.docs, start=1):
        score   = getattr(doc, "score",        "N/A")
        section = getattr(doc, "section_name", "")
        text    = getattr(doc, "text",         "")
        snippet = (text[:150].replace("\n", " ") + "…") if len(text) > 150 else text
        print(f"  #{rank}  score={score}  [{section}]")
        print(f"       {snippet}")
        print()
except Exception as exc:
    print(f"[warn] Filtered search failed (may be no matches): {exc}")

In [ ]:
# ── 15. Export embeddings to JSON (for offline inspection / backup) ────────
OUTPUT_JSON = "embedded_chunks_export.json"

export = []
for ec in embedded_chunks:
    export.append({
        "chunk_id":           ec.chunk_id,
        "source_document_id": ec.source_document_id,
        "section_name":       ec.section_name,
        "text":               ec.text,
        "embedding_model":    ec.embedding_model,
        "embedding":          ec.embedding,   # list of floats
        "metadata":           ec.metadata,
    })

with open(OUTPUT_JSON, "w", encoding="utf-8") as fh:
    json.dump(export, fh, ensure_ascii=False, default=str)

print(f"Exported {len(export):,} embedded chunks → {OUTPUT_JSON}")
print(f"File size: {Path(OUTPUT_JSON).stat().st_size / 1_048_576:.1f} MB")

In [ ]:
# ── 16. Pipeline summary ───────────────────────────────────────────────────
divider("PIPELINE SUMMARY")
print(f"  Source documents    : {len(documents):,}")
print(f"  Tables extracted    : {len(all_tables):,}  (persisted → {SQLITE_PATH})")
print(f"  Text chunks         : {len(chunks):,}")
print(f"  Embedded chunks     : {len(embedded_chunks):,}")
print(f"  Embedding model     : {EMBED_MODEL}  (dim={VECTOR_DIM})")
print(f"  Redis index         : {REDIS_INDEX}  ({REDIS_HOST}:{REDIS_PORT})")
print(f"  JSON backup         : {OUTPUT_JSON}")
print()
print("  Done. ✅")